In [1]:
from sklearn.model_selection import train_test_split
from pathlib import Path
from PIL import Image
import numpy as np

In [2]:
dataset_images_dir = Path("../dataset/CK+_prepared")
dataset_text_dir = Path("../dataset/CK+_coordinates")
dataset_cropped_dir = Path("../dataset/CK+_cropped")

In [3]:
dataset_224x224 = dataset_cropped_dir / "CK+_224x224"
dataset_224x224.mkdir(parents=True, exist_ok=True)

dataset_240x240 = dataset_cropped_dir / "CK+_240x240"
dataset_240x240.mkdir(parents=True, exist_ok=True)

dataset_288x288 = dataset_cropped_dir / "CK+_288x288"
dataset_288x288.mkdir(parents=True, exist_ok=True)

dataset_300x300 = dataset_cropped_dir / "CK+_300x300"
dataset_300x300.mkdir(parents=True, exist_ok=True)

dataset_380x380 = dataset_cropped_dir / "CK+_380x380"
dataset_380x380.mkdir(parents=True, exist_ok=True)

In [4]:
def copy_dataset(images_dict, output_dir):
    for class_name, image_paths in images_dict.items():
        class_dir = output_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        for image_path in image_paths:
            image_path["image"].save(class_dir / image_path["name"])

In [5]:
def dataset_splitting(input_dataset, output_path):
    train_path = output_path / "train"
    val_path = output_path / "val"
    test_path = output_path / "test"

    train_path.mkdir(parents=True, exist_ok=True)
    val_path.mkdir(parents=True, exist_ok=True)
    test_path.mkdir(parents=True, exist_ok=True)

    train_images = {}
    val_images = {}

    for class_name, image_paths in input_dataset.items():
        train, val = train_test_split(image_paths, test_size=0.20, shuffle=True, random_state=42)

        train_images[class_name] = train
        val_images[class_name] = val

    test_images = {}
    val_images_new = {}

    for class_name, image_paths in val_images.items():
        test, val = train_test_split(image_paths, test_size=0.50, shuffle=True, random_state=42)

        test_images[class_name] = test
        val_images_new[class_name] = val

    copy_dataset(train_images, train_path)
    copy_dataset(val_images_new, val_path)
    copy_dataset(test_images, test_path)

In [6]:
def transform_images(height_width):
    transformed_images = {}

    for class_dir in dataset_images_dir.iterdir():
        if not class_dir.is_dir():
            continue

        transformed_images[class_dir.name] = []

        coordinates_dir = dataset_text_dir / class_dir.name

        for image_path in class_dir.glob("*.png"):
            landmark_path = coordinates_dir / f"{image_path.stem}.txt"

            if not landmark_path.exists():
                print(f"Landmarks no encontrados: {image_path.name}")
                continue

            image = Image.open(image_path).convert("L").convert("RGB")

            landmarks = np.loadtxt(landmark_path)

            x = landmarks[:, 0]
            y = landmarks[:, 1]

            x_min = x.min()
            x_max = x.max()
            y_min = y.min()
            y_max = y.max()

            margin = 0.20

            width = x_max - x_min
            height = y_max - y_min

            x_min -= width * margin
            x_max += width * margin
            y_min -= height * margin
            y_max += height * margin

            center_x = (x_min + x_max) / 2
            center_y = (y_min + y_max) / 2

            crop_size = max(x_max - x_min, y_max - y_min)

            left = center_x - crop_size / 2
            right = center_x + crop_size / 2
            top = center_y - crop_size / 2
            bottom = center_y + crop_size / 2

            image_width, image_height = image.size

            if left < 0:
                right -= left
                left = 0

            if top < 0:
                bottom -= top
                top = 0

            if right > image_width:
                left -= right - image_width
                right = image_width

            if bottom > image_height:
                top -= bottom - image_height
                bottom = image_height

            left = int(round(left))
            top = int(round(top))
            right = int(round(right))
            bottom = int(round(bottom))

            crop = image.crop((left, top, right, bottom))

            crop = crop.resize((height_width, height_width), Image.Resampling.LANCZOS)

            transformed_images[class_dir.name].append({"image": crop, "name": image_path.name})

            print(f"{image_path.name} -> crop: {crop.size} -> {crop.size}")

    return transformed_images

In [7]:
images_224 = transform_images(224)
dataset_splitting(images_224, dataset_224x224)

images_240 = transform_images(240)
dataset_splitting(images_240, dataset_240x240)

images_288 = transform_images(288)
dataset_splitting(images_288, dataset_288x288)

images_300 = transform_images(300)
dataset_splitting(images_300, dataset_300x300)

images_380 = transform_images(380)
dataset_splitting(images_380, dataset_380x380)

S095_001.png -> crop: (224, 224) -> (224, 224)
S073_001.png -> crop: (224, 224) -> (224, 224)
S100_002.png -> crop: (224, 224) -> (224, 224)
S062_002.png -> crop: (224, 224) -> (224, 224)
S111_001.png -> crop: (224, 224) -> (224, 224)
S136_001.png -> crop: (224, 224) -> (224, 224)
S116_001.png -> crop: (224, 224) -> (224, 224)
S133_009.png -> crop: (224, 224) -> (224, 224)
S082_001.png -> crop: (224, 224) -> (224, 224)
S076_001.png -> crop: (224, 224) -> (224, 224)
S117_001.png -> crop: (224, 224) -> (224, 224)
S010_002.png -> crop: (224, 224) -> (224, 224)
S035_001.png -> crop: (224, 224) -> (224, 224)
S126_004.png -> crop: (224, 224) -> (224, 224)
S122_001.png -> crop: (224, 224) -> (224, 224)
S011_001.png -> crop: (224, 224) -> (224, 224)
S053_001.png -> crop: (224, 224) -> (224, 224)
S092_001.png -> crop: (224, 224) -> (224, 224)
S115_001.png -> crop: (224, 224) -> (224, 224)
S107_001.png -> crop: (224, 224) -> (224, 224)
S065_003.png -> crop: (224, 224) -> (224, 224)
S051_002.png 